# 02 — Data Pipeline 2024–2026

## Objective

Notebook ini digunakan untuk membangun pipeline otomatis yang memproses
NYC TLC HVFHV monthly data periode Januari 2024 hingga Mei 2026.

Pipeline akan:

1. Membentuk daftar monthly data Januari 2024–Mei 2026.
2. Memproses raw Parquet satu bulan pada satu waktu.
3. Menghasilkan monthly data-quality monitoring.
4. Menerapkan cleaning rules dari pilot Januari 2024.
5. Membentuk business analytics staging data.
6. Membentuk hourly demand staging data.
7. Menggabungkan seluruh monthly output.
8. Melakukan final pipeline validation sebelum data masuk BigQuery.

In [2]:
from pathlib import Path
from datetime import datetime

import pandas as pd
import duckdb
import requests

### Project & Folder Setup

In [3]:
print("Pandas :", pd.__version__)
print("DuckDB:", duckdb.__version__)

Pandas : 2.3.3
DuckDB: 1.5.5


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"

QUALITY_DIR = PROJECT_ROOT / "data" / "staging" / "quality"
BUSINESS_DIR = PROJECT_ROOT / "data" / "staging" / "business"
DEMAND_DIR = PROJECT_ROOT / "data" / "staging" / "demand"

OUTPUT_DIR = PROJECT_ROOT / "output"
SQL_DIR = PROJECT_ROOT / "sql"

folders = [
    RAW_DIR,
    QUALITY_DIR,
    BUSINESS_DIR,
    DEMAND_DIR,
    OUTPUT_DIR,
    SQL_DIR
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /Users/auliaaorama/Project/Ride hailing


Pada tahap awal pipeline, saya membuat struktur direktori yang memisahkan raw data, staging data, final output, dan SQL scripts. Pemisahan tersebut dilakukan agar raw source tetap terjaga, intermediate transformation dapat dilacak, dan final analytical output tidak bercampur dengan data mentah. Struktur folder juga dibuat secara programatis menggunakan pathlib sehingga pipeline lebih reproducible dan tidak bergantung pada absolute path.
```
STEP 1
Project & Folder Setup
        ↓
Menentukan PROJECT_ROOT
        ↓
Membuat folder:
raw
staging/quality
staging/business
staging/demand
output
sql
        ↓
Tidak ada data yang diolah
        ↓
Tujuannya:
rapi + reproducible + mudah dilacak
```

In [5]:
for folder in folders:
    print(folder)

/Users/auliaaorama/Project/Ride hailing/data/raw
/Users/auliaaorama/Project/Ride hailing/data/staging/quality
/Users/auliaaorama/Project/Ride hailing/data/staging/business
/Users/auliaaorama/Project/Ride hailing/data/staging/demand
/Users/auliaaorama/Project/Ride hailing/output
/Users/auliaaorama/Project/Ride hailing/sql


### menentukan periode 29 bulan dan membuat manifest pipeline.

In [6]:
START_MONTH = "2024-01"
END_MONTH = "2026-05"

months = pd.period_range(
    start=START_MONTH,
    end=END_MONTH,
    freq="M"
)

print("Jumlah bulan :", len(months))
print("Periode awal :", months[0])
print("Periode akhir:", months[-1])

Jumlah bulan : 29
Periode awal : 2024-01
Periode akhir: 2026-05


In [7]:
month_list = [str(month) for month in months]

month_list

['2024-01',
 '2024-02',
 '2024-03',
 '2024-04',
 '2024-05',
 '2024-06',
 '2024-07',
 '2024-08',
 '2024-09',
 '2024-10',
 '2024-11',
 '2024-12',
 '2025-01',
 '2025-02',
 '2025-03',
 '2025-04',
 '2025-05',
 '2025-06',
 '2025-07',
 '2025-08',
 '2025-09',
 '2025-10',
 '2025-11',
 '2025-12',
 '2026-01',
 '2026-02',
 '2026-03',
 '2026-04',
 '2026-05']

Saya mendefinisikan processing period secara programatis menggunakan pandas.period_range, yaitu Januari 2024 sampai Mei 2026 dengan monthly frequency. Pendekatan ini menghasilkan 29 monthly batches secara otomatis sehingga mengurangi risiko missing month atau kesalahan penulisan periode secara manual. Daftar periode tersebut kemudian digunakan sebagai control input untuk proses ingestion berikutnya

```
STEP 2
Define Processing Period

START_MONTH = 2024-01
END_MONTH   = 2026-05
        ↓
pd.period_range(freq="M")
        ↓
29 monthly periods
        ↓
convert ke string
        ↓
month_list
        ↓
digunakan untuk proses pipeline berikutnya

```

### Membentuk URL Data Secara Otomatis

In [8]:
BASE_URL = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data"
)

def build_tlc_url(month):
    filename = f"fhvhv_tripdata_{month}.parquet"
    return f"{BASE_URL}/{filename}"

In [9]:
print(build_tlc_url("2024-01"))
print(build_tlc_url("2026-05"))

https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-05.parquet


Setelah mendefinisikan monthly processing period, saya membuat URL generator berdasarkan pola nama file NYC TLC. Karena monthly files memiliki naming convention yang konsisten, URL dapat dibentuk secara dinamis dari parameter bulan. Pendekatan ini menghindari hardcoding 29 URL dan membuat ingestion process lebih scalable serta reproducible.

```
STEP 2
month_list
2024-01 → 2026-05
        ↓

STEP 3
Build URL

BASE_URL
+
fhvhv_tripdata_{month}.parquet
        ↓
Full monthly URL
        ↓
digunakan nanti oleh ingestion function
jika raw file belum tersedia
```

intinya: Mengubah setiap bulan menjadi alamat sumber file secara otomatis agar pipeline tidak bergantung pada download manual.

In [10]:
pipeline_manifest = pd.DataFrame({
    "month": month_list
})

pipeline_manifest["filename"] = (
    "fhvhv_tripdata_"
    + pipeline_manifest["month"]
    + ".parquet"
)

pipeline_manifest["url"] = (
    pipeline_manifest["month"]
    .apply(build_tlc_url)
)

pipeline_manifest.head()

,month,filename,url
0,2024-01,fhvhv_tripdata_2024-01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...
1,2024-02,fhvhv_tripdata_2024-02.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...
2,2024-03,fhvhv_tripdata_2024-03.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...
3,2024-04,fhvhv_tripdata_2024-04.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...
4,2024-05,fhvhv_tripdata_2024-05.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...


In [12]:
print("Total monthly batches:", len(pipeline_manifest))

Total monthly batches: 29


Setelah processing period dan source URL pattern ditentukan, saya membuat pipeline manifest menggunakan Pandas. Manifest berfungsi sebagai control table yang berisi monthly batch identifier, filename, dan source URL untuk seluruh 29 bulan. Dengan pendekatan tersebut, ingestion process dapat mengiterasi setiap batch secara sistematis tanpa hardcoding file satu per satu.

```
STEP 4
Create Pipeline Manifest

month_list
        ↓
buat DataFrame
        ↓
tambahkan filename
        ↓
tambahkan URL
        ↓

29 rows
1 row = 1 monthly batch

        ↓
digunakan sebagai daftar pekerjaan
untuk proses ingestion
```
Manifest adalah tabel kontrol yang memberi tahu pipeline bulan apa yang diproses, nama file apa yang digunakan, dan dari mana file tersebut dapat diperoleh



### TEST_MODE

memastikan pipeline kita tidak langsung memproses 29 bulan sekaligus saat kodenya masih baru.

In [14]:
TEST_MODE = True

if TEST_MODE:
    processing_manifest = pipeline_manifest.iloc[:1].copy()
else:
    processing_manifest = pipeline_manifest.copy()

processing_manifest

,month,filename,url
0,2024-01,fhvhv_tripdata_2024-01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...


### Check Local Raw File

In [15]:
test_row = processing_manifest.iloc[0]

test_month = test_row["month"]
test_filename = test_row["filename"]

test_file_path = RAW_DIR / test_filename

print("Month     :", test_month)
print("Filename  :", test_filename)
print("Local path:", test_file_path)

Month     : 2024-01
Filename  : fhvhv_tripdata_2024-01.parquet
Local path: /Users/auliaaorama/Project/Ride hailing/data/raw/fhvhv_tripdata_2024-01.parquet


In [16]:
file_exists = test_file_path.exists()

print("File exists:", file_exists)

File exists: True


In [17]:
if test_file_path.exists():
    print(f"[FOUND] {test_month} → gunakan file lokal")
else:
    print(f"[MISSING] {test_month} → file perlu didownload")

[FOUND] 2024-01 → gunakan file lokal


In [18]:
import requests

In [22]:
def ensure_local_file(month, filename, url):
    """
    Memastikan raw monthly Parquet tersedia di data/raw.

    Jika sudah tersedia, gunakan file lokal.
    Jika belum tersedia, download dari NYC TLC.
    """

    local_path = RAW_DIR / filename

    if local_path.exists():
        return local_path

    print(f"[DOWNLOAD] {month} → {filename}")

    temp_path = local_path.with_suffix(".parquet.part")

    try:
        with requests.get(
            url,
            stream=True,
            timeout=(30, 300)
        ) as response:

            response.raise_for_status()

            with open(temp_path, "wb") as file:
                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if chunk:
                        file.write(chunk)

        temp_path.replace(local_path)

        print(f"[SAVED] {month} → {filename}")

        return local_path

    except Exception:
        if temp_path.exists():
            temp_path.unlink()

        raise

### Read & Validate Monthly Parquet

Sebelum monthly file diproses, pipeline melakukan technical validation
untuk memastikan bahwa file dapat dibaca, tidak kosong, dan memiliki
kolom yang dibutuhkan untuk business analytics dan demand forecasting.

In [23]:
FORECAST_COLUMNS = [
    "hvfhs_license_num",
    "request_datetime",
    "PULocationID"
]

BUSINESS_COLUMNS = [
    "pickup_datetime",
    "dropoff_datetime",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "driver_pay",
    "shared_request_flag",
    "shared_match_flag",
    "wav_request_flag",
    "wav_match_flag"
]

REQUIRED_COLUMNS = list(
    dict.fromkeys(FORECAST_COLUMNS + BUSINESS_COLUMNS)
)

print("Jumlah required columns:", len(REQUIRED_COLUMNS))
REQUIRED_COLUMNS

Jumlah required columns: 14


['hvfhs_license_num',
 'request_datetime',
 'PULocationID',
 'pickup_datetime',
 'dropoff_datetime',
 'DOLocationID',
 'trip_miles',
 'trip_time',
 'base_passenger_fare',
 'driver_pay',
 'shared_request_flag',
 'shared_match_flag',
 'wav_request_flag',
 'wav_match_flag']

In [24]:
con = duckdb.connect()

In [25]:
def validate_monthly_file(file_path, month):
    """
    Memvalidasi monthly Parquet sebelum diproses lebih lanjut.

    Pemeriksaan:
    1. File dapat dibaca oleh DuckDB.
    2. File memiliki required columns.
    3. File memiliki minimal satu record.
    """

    file_path = Path(file_path)

    # ---------------------------------
    # 1. Read schema
    # ---------------------------------
    schema_df = con.sql(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{file_path.as_posix()}')
    """).df()

    available_columns = set(schema_df["column_name"])

    # ---------------------------------
    # 2. Check required columns
    # ---------------------------------
    missing_columns = [
        col
        for col in REQUIRED_COLUMNS
        if col not in available_columns
    ]

    if missing_columns:
        raise ValueError(
            f"{month}: required columns tidak ditemukan: "
            f"{missing_columns}"
        )

    # ---------------------------------
    # 3. Check total rows
    # ---------------------------------
    total_rows = con.sql(f"""
        SELECT COUNT(*) AS total_rows
        FROM read_parquet('{file_path.as_posix()}')
    """).fetchone()[0]

    if total_rows == 0:
        raise ValueError(
            f"{month}: file tidak memiliki data."
        )

    # ---------------------------------
    # Validation result
    # ---------------------------------
    result = {
        "month": month,
        "file_valid": True,
        "total_rows": total_rows,
        "total_columns": len(schema_df),
        "missing_required_columns": 0
    }

    return result

In [28]:
current_row = processing_manifest.iloc[0]

current_file = ensure_local_file(
    month=current_row["month"],
    filename=current_row["filename"],
    url=current_row["url"]
)

In [29]:
validation_result = validate_monthly_file(
    file_path=current_file,
    month=current_row["month"]
)

validation_result

{'month': '2024-01',
 'file_valid': True,
 'total_rows': 19663930,
 'total_columns': 24,
 'missing_required_columns': 0}

### Monthly Data Quality Monitoring

Setelah monthly file lolos technical validation, pipeline membuat
ringkasan data quality untuk memonitor konsistensi kualitas data
pada setiap bulan.

Pemeriksaan difokuskan pada variabel yang relevan terhadap
business analytics dan demand forecasting.

In [30]:
def get_monthly_quality(file_path, month):
    """
    Menghasilkan satu baris monthly data-quality summary.
    """

    file_path = Path(file_path)

    quality = con.sql(f"""
        SELECT
            COUNT(*) AS total_rows,

            -- Forecasting key variables
            SUM(
                CASE
                    WHEN request_datetime IS NULL
                    THEN 1 ELSE 0
                END
            ) AS missing_request_datetime,

            SUM(
                CASE
                    WHEN PULocationID IS NULL
                    THEN 1 ELSE 0
                END
            ) AS missing_PULocationID,

            SUM(
                CASE
                    WHEN PULocationID IS NOT NULL
                     AND (PULocationID < 1 OR PULocationID > 265)
                    THEN 1 ELSE 0
                END
            ) AS invalid_PULocationID,

            -- Optional operational variables
            SUM(
                CASE
                    WHEN originating_base_num IS NULL
                    THEN 1 ELSE 0
                END
            ) AS missing_originating_base,

            SUM(
                CASE
                    WHEN on_scene_datetime IS NULL
                    THEN 1 ELSE 0
                END
            ) AS missing_on_scene,

            -- Temporal quality
            SUM(
                CASE
                    WHEN pickup_datetime < request_datetime
                    THEN 1 ELSE 0
                END
            ) AS pickup_before_request,

            -- Numeric quality
            SUM(
                CASE
                    WHEN trip_miles = 0
                    THEN 1 ELSE 0
                END
            ) AS zero_trip_miles,

            SUM(
                CASE
                    WHEN trip_time = 0
                    THEN 1 ELSE 0
                END
            ) AS zero_trip_time,

            SUM(
                CASE
                    WHEN base_passenger_fare < 0
                    THEN 1 ELSE 0
                END
            ) AS negative_fare,

            SUM(
                CASE
                    WHEN driver_pay < 0
                    THEN 1 ELSE 0
                END
            ) AS negative_driver_pay

        FROM read_parquet('{file_path.as_posix()}')
    """).df()

    quality.insert(0, "month", month)

    return quality

In [31]:
monthly_quality = get_monthly_quality(
    file_path=current_file,
    month=current_row["month"]
)

monthly_quality

,month,total_rows,missing_request_datetime,missing_PULocationID,invalid_PULocationID,missing_originating_base,missing_on_scene,pickup_before_request,zero_trip_miles,zero_trip_time,negative_fare,negative_driver_pay
0,2024-01,19663930,0.0,0.0,0.0,5218737.0,5218737.0,188251.0,3286.0,2.0,191.0,137.0


In [34]:
monthly_quality["missing_request_pct"] = (
    monthly_quality["missing_request_datetime"]
    / monthly_quality["total_rows"]
    * 100
)

monthly_quality["invalid_PU_pct"] = (
    monthly_quality["invalid_PULocationID"]
    / monthly_quality["total_rows"]
    * 100
)

monthly_quality["pickup_before_request_pct"] = (
    monthly_quality["pickup_before_request"]
    / monthly_quality["total_rows"]
    * 100
)

monthly_quality["zero_miles_pct"] = (
    monthly_quality["zero_trip_miles"]
    / monthly_quality["total_rows"]
    * 100
)

monthly_quality["negative_fare_pct"] = (
    monthly_quality["negative_fare"]
    / monthly_quality["total_rows"]
    * 100
)


In [35]:
monthly_quality["missing_originating_base_pct"] = (
    monthly_quality["missing_originating_base"]
    / monthly_quality["total_rows"]
    * 100
)

monthly_quality["missing_on_scene_pct"] = (
    monthly_quality["missing_on_scene"]
    / monthly_quality["total_rows"]
    * 100
)

In [36]:
quality_view = monthly_quality[[
    "month",
    "total_rows",
    "missing_request_pct",
    "invalid_PU_pct",
    "missing_originating_base_pct",
    "missing_on_scene_pct",
    "pickup_before_request_pct",
    "zero_miles_pct",
    "negative_fare_pct"
]].copy()

pct_cols = [
    "missing_request_pct",
    "invalid_PU_pct",
    "missing_originating_base_pct",
    "missing_on_scene_pct",
    "pickup_before_request_pct",
    "zero_miles_pct",
    "negative_fare_pct"
]

quality_view[pct_cols] = quality_view[pct_cols].round(4)

quality_view

,month,total_rows,missing_request_pct,invalid_PU_pct,missing_originating_base_pct,missing_on_scene_pct,pickup_before_request_pct,zero_miles_pct,negative_fare_pct
0,2024-01,19663930,0.0,0.0,26.5396,26.5396,0.9573,0.0167,0.001




* `missing_request_pct = 0%` → tidak ada request time yang hilang, sehingga aman untuk pembentukan demand.
* `invalid_PU_pct = 0%` → seluruh pickup zone valid.
* `missing_originating_base_pct = 26,5396%` dan `missing_on_scene_pct = 26,5396%` → konsisten dengan hasil pilot; keduanya bukan variabel utama forecasting.
* `pickup_before_request_pct = 0,9573%` → sekitar 0,96% memiliki temporal inconsistency, tetapi tetap bisa dipakai untuk demand selama `request_datetime` dan pickup zone valid.
* `zero_miles_pct = 0,0167%` → sangat kecil dan hanya relevan untuk KPI jarak.
* `negative_fare_pct = 0,001%` → sangat kecil dan tidak digunakan untuk normal fare KPI.



### Apply Cleaning Rules

Cleaning rules dari pilot Januari 2024 diterapkan sebagai validity flags.
Pendekatan ini digunakan agar record yang tidak valid untuk satu KPI
tidak otomatis dihapus dari kebutuhan analisis lainnya.

In [37]:
def create_clean_base_view(file_path):
    """
    Membuat temporary DuckDB view dengan validity flags
    berdasarkan cleaning rules yang telah ditentukan.
    """

    file_path = Path(file_path)

    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW monthly_clean_base AS

        SELECT
            *,

            -- Valid untuk demand forecasting
            CASE
                WHEN request_datetime IS NOT NULL
                 AND PULocationID BETWEEN 1 AND 265
                THEN TRUE
                ELSE FALSE
            END AS is_forecast_valid,

            -- Valid untuk waiting-time analysis
            CASE
                WHEN request_datetime IS NOT NULL
                 AND pickup_datetime IS NOT NULL
                 AND pickup_datetime >= request_datetime
                THEN TRUE
                ELSE FALSE
            END AS is_waiting_time_valid,

            -- Valid untuk distance KPI
            CASE
                WHEN trip_miles > 0
                THEN TRUE
                ELSE FALSE
            END AS is_distance_valid,

            -- Valid untuk duration KPI
            CASE
                WHEN trip_time > 0
                THEN TRUE
                ELSE FALSE
            END AS is_duration_valid,

            -- Valid untuk normal passenger fare KPI
            CASE
                WHEN base_passenger_fare > 0
                THEN TRUE
                ELSE FALSE
            END AS is_fare_valid,

            -- Valid untuk normal driver pay KPI
            CASE
                WHEN driver_pay > 0
                THEN TRUE
                ELSE FALSE
            END AS is_driver_pay_valid

        FROM read_parquet('{file_path.as_posix()}')
    """)

In [38]:
create_clean_base_view(current_file)

In [39]:
cleaning_summary = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE WHEN is_forecast_valid
            THEN 1 ELSE 0 END
        ) AS forecast_valid_rows,

        SUM(
            CASE WHEN is_waiting_time_valid
            THEN 1 ELSE 0 END
        ) AS waiting_time_valid_rows,

        SUM(
            CASE WHEN is_distance_valid
            THEN 1 ELSE 0 END
        ) AS distance_valid_rows,

        SUM(
            CASE WHEN is_duration_valid
            THEN 1 ELSE 0 END
        ) AS duration_valid_rows,

        SUM(
            CASE WHEN is_fare_valid
            THEN 1 ELSE 0 END
        ) AS fare_valid_rows,

        SUM(
            CASE WHEN is_driver_pay_valid
            THEN 1 ELSE 0 END
        ) AS driver_pay_valid_rows

    FROM monthly_clean_base
""").df()

cleaning_summary

,total_rows,forecast_valid_rows,waiting_time_valid_rows,distance_valid_rows,duration_valid_rows,fare_valid_rows,driver_pay_valid_rows
0,19663930,19663930.0,19475679.0,19660644.0,19663928.0,19656596.0,19659297.0


### Business Transformation

Raw trip-level data ditransformasikan menjadi business analytics
staging data untuk mengurangi ukuran data sebelum dimuat ke BigQuery.

Main business staging menggunakan granularitas:

request hour × provider × pickup zone

Metric disimpan dalam bentuk additive measures (sum + valid count)
agar dapat diagregasi kembali secara akurat di BigQuery.

Pickup Hourly Business Metrics

In [40]:
def create_business_pickup_view():
    """
    Membuat business staging pada granularitas:
    request hour × provider × pickup zone.
    """

    con.execute("""
        CREATE OR REPLACE TEMP VIEW monthly_business_pickup AS

        SELECT
            DATE_TRUNC('hour', request_datetime) AS request_hour,

            hvfhs_license_num AS provider_code,

            CASE
                WHEN hvfhs_license_num = 'HV0003' THEN 'Uber'
                WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft'
                ELSE 'Other'
            END AS provider,

            CAST(PULocationID AS INTEGER) AS pickup_location_id,

            -- --------------------------------
            -- Trip volume
            -- --------------------------------
            COUNT(*) AS total_trips,

            -- --------------------------------
            -- Distance
            -- --------------------------------
            SUM(
                CASE
                    WHEN is_distance_valid
                    THEN trip_miles
                    ELSE 0
                END
            ) AS total_trip_miles,

            SUM(
                CASE
                    WHEN is_distance_valid
                    THEN 1 ELSE 0
                END
            ) AS valid_distance_trips,

            -- --------------------------------
            -- Duration
            -- --------------------------------
            SUM(
                CASE
                    WHEN is_duration_valid
                    THEN trip_time
                    ELSE 0
                END
            ) AS total_trip_seconds,

            SUM(
                CASE
                    WHEN is_duration_valid
                    THEN 1 ELSE 0
                END
            ) AS valid_duration_trips,

            -- --------------------------------
            -- Passenger fare
            -- --------------------------------
            SUM(
                CASE
                    WHEN is_fare_valid
                    THEN base_passenger_fare
                    ELSE 0
                END
            ) AS total_base_fare,

            SUM(
                CASE
                    WHEN is_fare_valid
                    THEN 1 ELSE 0
                END
            ) AS valid_fare_trips,

            -- --------------------------------
            -- Driver pay
            -- --------------------------------
            SUM(
                CASE
                    WHEN is_driver_pay_valid
                    THEN driver_pay
                    ELSE 0
                END
            ) AS total_driver_pay,

            SUM(
                CASE
                    WHEN is_driver_pay_valid
                    THEN 1 ELSE 0
                END
            ) AS valid_driver_pay_trips,

            -- --------------------------------
            -- Shared ride
            -- --------------------------------
            SUM(
                CASE
                    WHEN shared_request_flag = 'Y'
                    THEN 1 ELSE 0
                END
            ) AS shared_request_count,

            SUM(
                CASE
                    WHEN shared_match_flag = 'Y'
                    THEN 1 ELSE 0
                END
            ) AS shared_match_count,

            -- --------------------------------
            -- Wheelchair accessible vehicle
            -- --------------------------------
            SUM(
                CASE
                    WHEN wav_request_flag = 'Y'
                    THEN 1 ELSE 0
                END
            ) AS wav_request_count,

            SUM(
                CASE
                    WHEN wav_match_flag = 'Y'
                    THEN 1 ELSE 0
                END
            ) AS wav_match_count

        FROM monthly_clean_base

        WHERE is_forecast_valid = TRUE

        GROUP BY
            request_hour,
            provider_code,
            provider,
            pickup_location_id
    """)

Drop-off Daily Metrics

In [41]:
def create_business_dropoff_view():
    """
    Membuat staging destination analysis pada granularitas:
    request date × provider × drop-off zone.
    """

    con.execute("""
        CREATE OR REPLACE TEMP VIEW monthly_business_dropoff AS

        SELECT
            CAST(request_datetime AS DATE) AS request_date,

            hvfhs_license_num AS provider_code,

            CASE
                WHEN hvfhs_license_num = 'HV0003' THEN 'Uber'
                WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft'
                ELSE 'Other'
            END AS provider,

            CAST(DOLocationID AS INTEGER) AS dropoff_location_id,

            COUNT(*) AS total_dropoffs

        FROM monthly_clean_base

        WHERE is_forecast_valid = TRUE

        GROUP BY
            request_date,
            provider_code,
            provider,
            dropoff_location_id
    """)

In [42]:
create_business_pickup_view()
create_business_dropoff_view()

In [43]:
business_transform_summary = con.sql("""
    SELECT
        (SELECT COUNT(*)
         FROM monthly_clean_base)
            AS raw_rows,

        (SELECT COUNT(*)
         FROM monthly_business_pickup)
            AS pickup_staging_rows,

        (SELECT COUNT(*)
         FROM monthly_business_dropoff)
            AS dropoff_staging_rows
""").df()

business_transform_summary

,raw_rows,pickup_staging_rows,dropoff_staging_rows
0,19663930,365552,16371


Business transformation berhasil mengurangi granularitas raw trip-level data secara signifikan. Dari 19.663.930 record perjalanan Januari 2024, agregasi berdasarkan request_hour × provider × pickup_zone menghasilkan 365.552 record, sedangkan agregasi request_date × provider × dropoff_zone menghasilkan 16.371 record. Reduksi ini memungkinkan historical business analytics tetap mempertahankan informasi utama seperti volume perjalanan, jarak, durasi, fare, driver pay, shared ride, dan WAV metrics, tanpa harus menyimpan seluruh individual trip di analytical warehouse.

### Hourly Demand Transformation

Forecasting dataset dibentuk dengan mengagregasi valid ride request
pada granularitas:

request hour × provider × pickup zone

Target `demand` merepresentasikan jumlah customer request pada
kombinasi waktu dan area tertentu.

In [49]:
def create_hourly_demand_view():
    """
    Membentuk monthly hourly demand pada granularitas:
    request hour × provider × pickup zone.
    """

    con.execute("""
        CREATE OR REPLACE TEMP VIEW monthly_hourly_demand AS

        SELECT
            DATE_TRUNC(
                'hour',
                request_datetime
            ) AS request_hour,

            hvfhs_license_num AS provider_code,

            CASE
                WHEN hvfhs_license_num = 'HV0003' THEN 'Uber'
                WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft'
                ELSE 'Other'
            END AS provider,

            CAST(PULocationID AS INTEGER) AS pickup_location_id,

            COUNT(*) AS demand

        FROM monthly_clean_base

        WHERE is_forecast_valid = TRUE

        GROUP BY
            request_hour,
            provider_code,
            provider,
            pickup_location_id
    """)

In [50]:
create_hourly_demand_view()

In [51]:
hourly_demand_preview = con.sql("""
    SELECT *
    FROM monthly_hourly_demand
    ORDER BY
        request_hour,
        provider_code,
        pickup_location_id
    LIMIT 10
""").df()

hourly_demand_preview

,request_hour,provider_code,provider,pickup_location_id,demand
0,2023-12-31 23:00:00,HV0003,Uber,3,1
1,2023-12-31 23:00:00,HV0003,Uber,4,1
2,2023-12-31 23:00:00,HV0003,Uber,5,1
3,2023-12-31 23:00:00,HV0003,Uber,6,1
4,2023-12-31 23:00:00,HV0003,Uber,9,1
5,2023-12-31 23:00:00,HV0003,Uber,10,2
6,2023-12-31 23:00:00,HV0003,Uber,13,1
7,2023-12-31 23:00:00,HV0003,Uber,14,1
8,2023-12-31 23:00:00,HV0003,Uber,16,2
9,2023-12-31 23:00:00,HV0003,Uber,17,2


In [52]:
hourly_demand_summary = con.sql("""
    SELECT
        COUNT(*) AS hourly_demand_rows,

        SUM(demand) AS total_demand,

        MIN(request_hour) AS min_request_hour,

        MAX(request_hour) AS max_request_hour,

        MIN(demand) AS min_demand,

        MAX(demand) AS max_demand

    FROM monthly_hourly_demand
""").df()

hourly_demand_summary

,hourly_demand_rows,total_demand,min_request_hour,max_request_hour,min_demand,max_demand
0,365552,19663930.0,2023-12-31 23:00:00,2024-02-01,1,1266


### Save Monthly Outputs

In [53]:
current_month = current_row["month"]

current_month

'2024-01'

In [54]:
quality_output = (
    QUALITY_DIR
    / f"quality_{current_month}.csv"
)

monthly_quality.to_csv(
    quality_output,
    index=False
)

print("Saved:", quality_output)

Saved: /Users/auliaaorama/Project/Ride hailing/data/staging/quality/quality_2024-01.csv


In [55]:
business_pickup_output = (
    BUSINESS_DIR
    / f"business_pickup_{current_month}.parquet"
)

con.execute(f"""
    COPY (
        SELECT
            '{current_month}' AS source_month,
            *
        FROM monthly_business_pickup
    )
    TO '{business_pickup_output.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", business_pickup_output)

Saved: /Users/auliaaorama/Project/Ride hailing/data/staging/business/business_pickup_2024-01.parquet


In [56]:
business_dropoff_output = (
    BUSINESS_DIR
    / f"business_dropoff_{current_month}.parquet"
)

con.execute(f"""
    COPY (
        SELECT
            '{current_month}' AS source_month,
            *
        FROM monthly_business_dropoff
    )
    TO '{business_dropoff_output.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", business_dropoff_output)

Saved: /Users/auliaaorama/Project/Ride hailing/data/staging/business/business_dropoff_2024-01.parquet


In [57]:
demand_output = (
    DEMAND_DIR
    / f"hourly_demand_{current_month}.parquet"
)

con.execute(f"""
    COPY (
        SELECT
            '{current_month}' AS source_month,
            *
        FROM monthly_hourly_demand
    )
    TO '{demand_output.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", demand_output)

Saved: /Users/auliaaorama/Project/Ride hailing/data/staging/demand/hourly_demand_2024-01.parquet


In [58]:
saved_outputs = {
    "quality": quality_output,
    "business_pickup": business_pickup_output,
    "business_dropoff": business_dropoff_output,
    "hourly_demand": demand_output
}

for name, path in saved_outputs.items():
    print(
        f"{name:20s}",
        "OK" if path.exists() else "MISSING",
        path.name
    )

quality              OK quality_2024-01.csv
business_pickup      OK business_pickup_2024-01.parquet
business_dropoff     OK business_dropoff_2024-01.parquet
hourly_demand        OK hourly_demand_2024-01.parquet


### Automate Monthly Pipeline

Fungsi Persentase Quality Report

In [59]:
def add_quality_percentages(quality_df):
    """
    Menambahkan persentase data-quality agar
    hasil antarbulan dapat dibandingkan.
    """

    df = quality_df.copy()

    df["missing_request_pct"] = (
        df["missing_request_datetime"]
        / df["total_rows"]
        * 100
    )

    df["missing_PU_pct"] = (
        df["missing_PULocationID"]
        / df["total_rows"]
        * 100
    )

    df["invalid_PU_pct"] = (
        df["invalid_PULocationID"]
        / df["total_rows"]
        * 100
    )

    df["missing_originating_base_pct"] = (
        df["missing_originating_base"]
        / df["total_rows"]
        * 100
    )

    df["missing_on_scene_pct"] = (
        df["missing_on_scene"]
        / df["total_rows"]
        * 100
    )

    df["pickup_before_request_pct"] = (
        df["pickup_before_request"]
        / df["total_rows"]
        * 100
    )

    df["zero_miles_pct"] = (
        df["zero_trip_miles"]
        / df["total_rows"]
        * 100
    )

    df["zero_time_pct"] = (
        df["zero_trip_time"]
        / df["total_rows"]
        * 100
    )

    df["negative_fare_pct"] = (
        df["negative_fare"]
        / df["total_rows"]
        * 100
    )

    df["negative_driver_pay_pct"] = (
        df["negative_driver_pay"]
        / df["total_rows"]
        * 100
    )

    return df

Fungsi untuk Menentukan Lokasi Output

In [60]:
def get_monthly_output_paths(month):
    """
    Menentukan lokasi seluruh output untuk satu bulan.
    """

    return {
        "quality": (
            QUALITY_DIR
            / f"quality_{month}.csv"
        ),

        "business_pickup": (
            BUSINESS_DIR
            / f"business_pickup_{month}.parquet"
        ),

        "business_dropoff": (
            BUSINESS_DIR
            / f"business_dropoff_{month}.parquet"
        ),

        "hourly_demand": (
            DEMAND_DIR
            / f"hourly_demand_{month}.parquet"
        )
    }

In [61]:
get_monthly_output_paths("2024-01")

{'quality': PosixPath('/Users/auliaaorama/Project/Ride hailing/data/staging/quality/quality_2024-01.csv'),
 'business_pickup': PosixPath('/Users/auliaaorama/Project/Ride hailing/data/staging/business/business_pickup_2024-01.parquet'),
 'business_dropoff': PosixPath('/Users/auliaaorama/Project/Ride hailing/data/staging/business/business_dropoff_2024-01.parquet'),
 'hourly_demand': PosixPath('/Users/auliaaorama/Project/Ride hailing/data/staging/demand/hourly_demand_2024-01.parquet')}

Fungsi Save

In [62]:
def save_monthly_outputs(month, monthly_quality):
    """
    Menyimpan seluruh staging output satu bulan.
    """

    paths = get_monthly_output_paths(month)

    # -----------------------------
    # Quality report
    # -----------------------------
    monthly_quality.to_csv(
        paths["quality"],
        index=False
    )

    # -----------------------------
    # Business pickup
    # -----------------------------
    con.execute(f"""
        COPY (
            SELECT
                '{month}' AS source_month,
                *
            FROM monthly_business_pickup
        )
        TO '{paths["business_pickup"].as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    # -----------------------------
    # Business dropoff
    # -----------------------------
    con.execute(f"""
        COPY (
            SELECT
                '{month}' AS source_month,
                *
            FROM monthly_business_dropoff
        )
        TO '{paths["business_dropoff"].as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    # -----------------------------
    # Hourly demand
    # -----------------------------
    con.execute(f"""
        COPY (
            SELECT
                '{month}' AS source_month,
                *
            FROM monthly_hourly_demand
        )
        TO '{paths["hourly_demand"].as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    return paths

In [63]:
def process_month(row):
    """
    Menjalankan seluruh pipeline untuk satu monthly batch.
    """

    month = row["month"]
    filename = row["filename"]
    url = row["url"]

    print("\n" + "=" * 60)
    print(f"PROCESSING MONTH: {month}")
    print("=" * 60)

    # -----------------------------------
    # 1. Ensure raw file tersedia
    # -----------------------------------
    file_path = ensure_local_file(
        month=month,
        filename=filename,
        url=url
    )

    # -----------------------------------
    # 2. Technical validation
    # -----------------------------------
    validation = validate_monthly_file(
        file_path=file_path,
        month=month
    )

    print(
        f"[VALID] {month} → "
        f"{validation['total_rows']:,} raw rows"
    )

    # -----------------------------------
    # 3. Monthly data quality monitoring
    # -----------------------------------
    monthly_quality = get_monthly_quality(
        file_path=file_path,
        month=month
    )

    monthly_quality = add_quality_percentages(
        monthly_quality
    )

    # -----------------------------------
    # 4. Apply cleaning rules
    # -----------------------------------
    create_clean_base_view(file_path)

    # -----------------------------------
    # 5. Business transformation
    # -----------------------------------
    create_business_pickup_view()
    create_business_dropoff_view()

    # -----------------------------------
    # 6. Hourly demand transformation
    # -----------------------------------
    create_hourly_demand_view()

    # -----------------------------------
    # 7. Pipeline control check
    # -----------------------------------
    control = con.sql("""
        SELECT
            (
                SELECT COUNT(*)
                FROM monthly_clean_base
            ) AS raw_rows,

            (
                SELECT SUM(
                    CASE
                        WHEN is_forecast_valid
                        THEN 1 ELSE 0
                    END
                )
                FROM monthly_clean_base
            ) AS forecast_valid_rows,

            (
                SELECT COUNT(*)
                FROM monthly_business_pickup
            ) AS business_pickup_rows,

            (
                SELECT COUNT(*)
                FROM monthly_business_dropoff
            ) AS business_dropoff_rows,

            (
                SELECT COUNT(*)
                FROM monthly_hourly_demand
            ) AS hourly_demand_rows,

            (
                SELECT SUM(demand)
                FROM monthly_hourly_demand
            ) AS total_demand
    """).df()

    # -----------------------------------
    # 8. Validate demand aggregation
    # -----------------------------------
    forecast_valid = int(
        control.loc[0, "forecast_valid_rows"]
    )

    total_demand = int(
        control.loc[0, "total_demand"]
    )

    if forecast_valid != total_demand:
        raise ValueError(
            f"{month}: total demand tidak sama "
            f"dengan valid request."
        )

    # -----------------------------------
    # 9. Save monthly output
    # -----------------------------------
    paths = save_monthly_outputs(
        month=month,
        monthly_quality=monthly_quality
    )

    # -----------------------------------
    # 10. Return pipeline summary
    # -----------------------------------
    result = {
        "month": month,
        "status": "SUCCESS",

        "raw_rows": int(
            control.loc[0, "raw_rows"]
        ),

        "forecast_valid_rows": forecast_valid,

        "business_pickup_rows": int(
            control.loc[0, "business_pickup_rows"]
        ),

        "business_dropoff_rows": int(
            control.loc[0, "business_dropoff_rows"]
        ),

        "hourly_demand_rows": int(
            control.loc[0, "hourly_demand_rows"]
        ),

        "total_demand": total_demand
    }

    print(f"[SUCCESS] {month}")

    return result

In [64]:
jan_paths = get_monthly_output_paths("2024-01")

for path in jan_paths.values():
    if path.exists():
        path.unlink()

In [65]:
test_result = process_month(
    processing_manifest.iloc[0]
)

test_result


PROCESSING MONTH: 2024-01
[VALID] 2024-01 → 19,663,930 raw rows
[SUCCESS] 2024-01


{'month': '2024-01',
 'status': 'SUCCESS',
 'raw_rows': 19663930,
 'forecast_valid_rows': 19663930,
 'business_pickup_rows': 365552,
 'business_dropoff_rows': 16371,
 'hourly_demand_rows': 365552,
 'total_demand': 19663930}

In [66]:
{
    'month': '2024-01',
    'status': 'SUCCESS',
    'raw_rows': 19663930,
    'forecast_valid_rows': 19663930,
    'business_pickup_rows': 365552,
    'business_dropoff_rows': 16371,
    'hourly_demand_rows': 365552,
    'total_demand': 19663930
}

{'month': '2024-01',
 'status': 'SUCCESS',
 'raw_rows': 19663930,
 'forecast_valid_rows': 19663930,
 'business_pickup_rows': 365552,
 'business_dropoff_rows': 16371,
 'hourly_demand_rows': 365552,
 'total_demand': 19663930}

In [67]:
TEST_MODE = False

if TEST_MODE:
    processing_manifest = (
        pipeline_manifest.iloc[:1].copy()
    )
else:
    processing_manifest = (
        pipeline_manifest.copy()
    )

print(
    "Jumlah bulan yang akan diproses:",
    len(processing_manifest)
)

Jumlah bulan yang akan diproses: 29


In [68]:
pipeline_results = []

for _, row in processing_manifest.iterrows():

    try:
        result = process_month(row)

    except Exception as error:

        print(
            f"[FAILED] {row['month']} → {error}"
        )

        result = {
            "month": row["month"],
            "status": "FAILED",
            "error": str(error)
        }

    pipeline_results.append(result)


PROCESSING MONTH: 2024-01
[VALID] 2024-01 → 19,663,930 raw rows
[SUCCESS] 2024-01

PROCESSING MONTH: 2024-02
[DOWNLOAD] 2024-02 → fhvhv_tripdata_2024-02.parquet
[SAVED] 2024-02 → fhvhv_tripdata_2024-02.parquet
[VALID] 2024-02 → 19,359,148 raw rows
[SUCCESS] 2024-02

PROCESSING MONTH: 2024-03
[DOWNLOAD] 2024-03 → fhvhv_tripdata_2024-03.parquet
[SAVED] 2024-03 → fhvhv_tripdata_2024-03.parquet
[VALID] 2024-03 → 21,280,788 raw rows
[SUCCESS] 2024-03

PROCESSING MONTH: 2024-04
[DOWNLOAD] 2024-04 → fhvhv_tripdata_2024-04.parquet
[SAVED] 2024-04 → fhvhv_tripdata_2024-04.parquet
[VALID] 2024-04 → 19,733,038 raw rows
[SUCCESS] 2024-04

PROCESSING MONTH: 2024-05
[DOWNLOAD] 2024-05 → fhvhv_tripdata_2024-05.parquet
[SAVED] 2024-05 → fhvhv_tripdata_2024-05.parquet
[VALID] 2024-05 → 20,704,538 raw rows
[SUCCESS] 2024-05

PROCESSING MONTH: 2024-06
[DOWNLOAD] 2024-06 → fhvhv_tripdata_2024-06.parquet
[SAVED] 2024-06 → fhvhv_tripdata_2024-06.parquet
[VALID] 2024-06 → 20,123,226 raw rows
[SUCCESS] 2024-

In [69]:
pipeline_run_report = pd.DataFrame(
    pipeline_results
)

pipeline_run_report

,month,status,raw_rows,forecast_valid_rows,business_pickup_rows,business_dropoff_rows,hourly_demand_rows,total_demand
0,2024-01,SUCCESS,19663930,19663930,365552,16371,365552,19663930
1,2024-02,SUCCESS,19359148,19359148,342476,15398,342476,19359148
2,2024-03,SUCCESS,21280788,21280788,366441,16481,366441,21280788
3,2024-04,SUCCESS,19733038,19733038,355359,15937,355359,19733038
4,2024-05,SUCCESS,20704538,20704538,367865,16456,367865,20704538
5,2024-06,SUCCESS,20123226,20123226,357262,15992,357262,20123226
6,2024-07,SUCCESS,19182934,19182934,368644,16502,368644,19182934
7,2024-08,SUCCESS,19128392,19128392,368584,16461,368584,19128392
8,2024-09,SUCCESS,19209788,19209788,355464,15984,355464,19209788
9,2024-10,SUCCESS,20028282,20028282,366711,16415,366711,20028282


### Folder Final Output

In [70]:
FINAL_DIR = OUTPUT_DIR / "final"

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Final output directory:")
print(FINAL_DIR)

Final output directory:
/Users/auliaaorama/Project/Ride hailing/output/final


### Gabungkan Monthly Quality Report

In [71]:
quality_files = sorted(
    QUALITY_DIR.glob("quality_*.csv")
)

print(
    "Jumlah quality files:",
    len(quality_files)
)

Jumlah quality files: 29


In [72]:
monthly_quality_final = pd.concat(
    [
        pd.read_csv(file)
        for file in quality_files
    ],
    ignore_index=True
)

monthly_quality_final = (
    monthly_quality_final
    .sort_values("month")
    .reset_index(drop=True)
)

print(
    "Jumlah bulan:",
    len(monthly_quality_final)
)

monthly_quality_final[
    ["month", "total_rows"]
]

Jumlah bulan: 29


,month,total_rows
0,2024-01,19663930
1,2024-02,19359148
2,2024-03,21280788
3,2024-04,19733038
4,2024-05,20704538
5,2024-06,20123226
6,2024-07,19182934
7,2024-08,19128392
8,2024-09,19209788
9,2024-10,20028282


In [73]:
quality_final_path = (
    FINAL_DIR
    / "monthly_data_quality_2024_2026.csv"
)

monthly_quality_final.to_csv(
    quality_final_path,
    index=False
)

print("Saved:", quality_final_path)

Saved: /Users/auliaaorama/Project/Ride hailing/output/final/monthly_data_quality_2024_2026.csv


### Gabungkan Final Hourly Demand

In [74]:
demand_pattern = (
    DEMAND_DIR
    / "hourly_demand_*.parquet"
)

demand_pattern

PosixPath('/Users/auliaaorama/Project/Ride hailing/data/staging/demand/hourly_demand_*.parquet')

In [75]:
final_demand_path = (
    FINAL_DIR
    / "hourly_demand_2024_2026.parquet"
)

Transformation final

In [76]:
con.execute(f"""
    COPY (

        SELECT
            request_hour,
            provider_code,
            provider,
            pickup_location_id,

            SUM(demand) AS demand

        FROM read_parquet(
            '{demand_pattern.as_posix()}'
        )

        WHERE request_hour >=
            TIMESTAMP '2024-01-01 00:00:00'

          AND request_hour <
            TIMESTAMP '2026-06-01 00:00:00'

        GROUP BY
            request_hour,
            provider_code,
            provider,
            pickup_location_id

        ORDER BY
            request_hour,
            provider_code,
            pickup_location_id

    )
    TO '{final_demand_path.as_posix()}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print("Saved:", final_demand_path)

Saved: /Users/auliaaorama/Project/Ride hailing/output/final/hourly_demand_2024_2026.parquet


### Gabungkan Business Pickup

In [77]:
business_pickup_pattern = (
    BUSINESS_DIR
    / "business_pickup_*.parquet"
)

final_business_pickup_path = (
    FINAL_DIR
    / "business_pickup_2024_2026.parquet"
)

In [78]:
con.execute(f"""
    COPY (

        SELECT
            request_hour,
            provider_code,
            provider,
            pickup_location_id,

            SUM(total_trips)
                AS total_trips,

            SUM(total_trip_miles)
                AS total_trip_miles,

            SUM(valid_distance_trips)
                AS valid_distance_trips,

            SUM(total_trip_seconds)
                AS total_trip_seconds,

            SUM(valid_duration_trips)
                AS valid_duration_trips,

            SUM(total_base_fare)
                AS total_base_fare,

            SUM(valid_fare_trips)
                AS valid_fare_trips,

            SUM(total_driver_pay)
                AS total_driver_pay,

            SUM(valid_driver_pay_trips)
                AS valid_driver_pay_trips,

            SUM(shared_request_count)
                AS shared_request_count,

            SUM(shared_match_count)
                AS shared_match_count,

            SUM(wav_request_count)
                AS wav_request_count,

            SUM(wav_match_count)
                AS wav_match_count

        FROM read_parquet(
            '{business_pickup_pattern.as_posix()}'
        )

        WHERE request_hour >=
            TIMESTAMP '2024-01-01 00:00:00'

          AND request_hour <
            TIMESTAMP '2026-06-01 00:00:00'

        GROUP BY
            request_hour,
            provider_code,
            provider,
            pickup_location_id

        ORDER BY
            request_hour,
            provider_code,
            pickup_location_id

    )
    TO '{final_business_pickup_path.as_posix()}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print(
    "Saved:",
    final_business_pickup_path
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: /Users/auliaaorama/Project/Ride hailing/output/final/business_pickup_2024_2026.parquet


### Gabungkan Business Drop-off

In [79]:
business_dropoff_pattern = (
    BUSINESS_DIR
    / "business_dropoff_*.parquet"
)

final_business_dropoff_path = (
    FINAL_DIR
    / "business_dropoff_2024_2026.parquet"
)

In [80]:
con.execute(f"""
    COPY (

        SELECT
            request_date,
            provider_code,
            provider,
            dropoff_location_id,

            SUM(total_dropoffs)
                AS total_dropoffs

        FROM read_parquet(
            '{business_dropoff_pattern.as_posix()}'
        )

        WHERE request_date >= DATE '2024-01-01'
          AND request_date <  DATE '2026-06-01'

        GROUP BY
            request_date,
            provider_code,
            provider,
            dropoff_location_id

        ORDER BY
            request_date,
            provider_code,
            dropoff_location_id

    )
    TO '{final_business_dropoff_path.as_posix()}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print(
    "Saved:",
    final_business_dropoff_path
)

Saved: /Users/auliaaorama/Project/Ride hailing/output/final/business_dropoff_2024_2026.parquet


In [81]:
final_outputs = {
    "quality":
        quality_final_path,

    "business_pickup":
        final_business_pickup_path,

    "business_dropoff":
        final_business_dropoff_path,

    "hourly_demand":
        final_demand_path
}

for name, path in final_outputs.items():
    print(
        f"{name:20s}",
        "OK" if path.exists() else "MISSING",
        path.name
    )

quality              OK monthly_data_quality_2024_2026.csv
business_pickup      OK business_pickup_2024_2026.parquet
business_dropoff     OK business_dropoff_2024_2026.parquet
hourly_demand        OK hourly_demand_2024_2026.parquet


### Validasi Quality Report

In [82]:
quality_validation = pd.read_csv(
    FINAL_DIR / "monthly_data_quality_2024_2026.csv"
)

print("Jumlah bulan:", len(quality_validation))
print("Periode awal:", quality_validation["month"].min())
print("Periode akhir:", quality_validation["month"].max())

Jumlah bulan: 29
Periode awal: 2024-01
Periode akhir: 2026-05


### Validasi Final Hourly Demand

In [83]:
final_demand_validation = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        MIN(request_hour) AS min_request_hour,
        MAX(request_hour) AS max_request_hour,

        MIN(demand) AS min_demand,
        MAX(demand) AS max_demand,

        SUM(demand) AS total_demand,

        COUNT(DISTINCT provider_code) AS total_providers,

        MIN(pickup_location_id) AS min_pickup_zone,
        MAX(pickup_location_id) AS max_pickup_zone

    FROM read_parquet(
        '{final_demand_path.as_posix()}'
    )
""").df()

final_demand_validation

,total_rows,min_request_hour,max_request_hour,min_demand,max_demand,total_demand,total_providers,min_pickup_zone,max_pickup_zone
0,10446585,2024-01-01,2026-05-31 23:00:00,1.0,2200.0,589055372.0,2,1,265


### Cek Duplicate Key Forecasting

In [84]:
duplicate_demand_key = con.sql(f"""
    SELECT
        COUNT(*) AS duplicate_groups

    FROM (
        SELECT
            request_hour,
            provider_code,
            pickup_location_id,
            COUNT(*) AS row_count

        FROM read_parquet(
            '{final_demand_path.as_posix()}'
        )

        GROUP BY
            request_hour,
            provider_code,
            pickup_location_id

        HAVING COUNT(*) > 1
    )
""").df()

duplicate_demand_key

,duplicate_groups
0,0


### Cek Provider

In [85]:
provider_validation = con.sql(f"""
    SELECT
        provider_code,
        provider,
        SUM(demand) AS total_demand

    FROM read_parquet(
        '{final_demand_path.as_posix()}'
    )

    GROUP BY
        provider_code,
        provider

    ORDER BY total_demand DESC
""").df()

provider_validation

,provider_code,provider,total_demand
0,HV0003,Uber,431217282.0
1,HV0005,Lyft,157838090.0
